# Sentinel-1 before/after imagery for the 2026 Nepal landslide

Search the Copernicus Data Space STAC catalog for Sentinel-1 SLC acquisitions around the 26 August 2026 Nepal–Tibet border landslide. The selection keeps before/after scenes on the same relative orbit and pass direction so they are suitable for comparison.

The event date in the GeoJSON has day precision. Acquisitions on that date are therefore shown separately and are not automatically treated as before or after. The notebook automatically selects the two closest pre-event scenes—one control pair—and the closest post-event scenes; if no unambiguous post-event acquisition has been published yet, it can simply be rerun later.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import logging

import geopandas as gpd
import pandas as pd

from eo_tools.S1.download import download_partial_products, search_products
from eo_tools.util import explore_products
from eo_tools_dev.util import serve_map

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

## Configuration

In [ ]:
aoi_file = Path("../data/nepal_landslide_bounding_box.geojson")
data_dir = Path("/data/S1/partial_dls/Nepal_landslide_2026")
output_root = Path("/data/res/nepal-landslide-s1")
credentials_file = Path("/data/creds_s3.json")

days_before = 45
days_after = 30
pre_event_scenes = 2
post_event_scenes = 2

output_root.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)

## Read the event footprint and date

In [ ]:
event = gpd.read_file(aoi_file).to_crs("EPSG:4326")
if len(event) != 1 or event.geometry.iloc[0].geom_type != "Polygon":
    raise ValueError("Expected exactly one Polygon feature in the event GeoJSON")

shp = event.geometry.iloc[0]
event_day = pd.Timestamp(event.iloc[0]["date"])
search_start = event_day - pd.Timedelta(days=days_before)
search_end = event_day + pd.Timedelta(days=days_after)

print(event.iloc[0]["name"])
print(f"Event date: {event_day.date()}")
print(f"Search window: {search_start.date()} to {search_end.date()}")
event.explore(tooltip=["name", "date", "description"] )

## Search Sentinel-1 SLC products

In [ ]:
products = search_products(
    intersects=shp,
    datetime=[search_start.strftime("%Y-%m-%d"), search_end.strftime("%Y-%m-%d")],
)

if products.empty:
    raise RuntimeError("No Sentinel-1 SLC products found in the search window")

products = products.set_crs("EPSG:4326", allow_override=True).copy()
products["acquired"] = pd.to_datetime(
    products["startTimeFromAscendingNode"], utc=True
).dt.tz_localize(None)

event_day_end = event_day + pd.Timedelta(days=1)
products["period"] = "event-day (review manually)"
products.loc[products.acquired < event_day, "period"] = "before"
products.loc[products.acquired >= event_day_end, "period"] = "after"
products = products.sort_values("acquired").reset_index(drop=True)

display(products[["acquired", "period", "relativeOrbitNumber", "orbitDirection", "id"]])
display(
    products.groupby(
        ["relativeOrbitNumber", "orbitDirection", "period"]
    ).size().rename("scene_count").to_frame()
)
products.drop(columns="stac_item").to_file(
    output_root / "products.geojson", driver="GeoJSON"
)

In [ ]:
m = explore_products(products=products, aoi=shp)
serve_map(m)

## Select a comparable orbit and the closest scenes

Prefer an orbit/pass with acquisitions on both sides of the event. Select the two closest pre-event scenes and up to `post_event_scenes` closest post-event scenes from that orbit/pass. The two pre-event scenes provide a control pair; the closest pre-event scene is also paired with each post-event scene. Until a post-event scene exists, use the orbit with the closest pre-event coverage. Event-day scenes remain in the table above for manual review but are excluded from automatic selection because the event time is unknown.

In [ ]:
# Defaults make this cell safe to rerun without rerunning Configuration first.
pre_event_scenes = globals().get("pre_event_scenes", 2)
post_event_scenes = globals().get("post_event_scenes", 2)

unambiguous = products[products.period.isin(["before", "after"])].copy()
if unambiguous.empty:
    raise RuntimeError("Only event-day products were found; review them manually")

orbit_cols = ["relativeOrbitNumber", "orbitDirection"]
orbit_options = []
for orbit_key, group in unambiguous.groupby(orbit_cols):
    before = group[group.period == "before"]
    after = group[group.period == "after"]
    if len(before) < pre_event_scenes:
        continue
    orbit_options.append(
        {
            "orbit_key": orbit_key,
            "has_after": not after.empty,
            "before_gap": (event_day - before.acquired.max()).total_seconds(),
            "after_gap": (after.acquired.min() - event_day_end).total_seconds() if not after.empty else float("inf"),
            "count": len(group),
        }
    )

if not orbit_options:
    raise RuntimeError(
        f"No orbit/pass has at least {pre_event_scenes} pre-event acquisitions"
    )

best = sorted(
    orbit_options,
    key=lambda row: (not row["has_after"], row["before_gap"] + row["after_gap"], row["before_gap"], -row["count"]),
)[0]
relative_orbit, pass_direction = best["orbit_key"]
same_orbit = unambiguous[
    (unambiguous.relativeOrbitNumber == relative_orbit)
    & (unambiguous.orbitDirection == pass_direction)
].copy()

selected_before = same_orbit[same_orbit.period == "before"].nlargest(
    pre_event_scenes, "acquired"
)
selected_after = same_orbit[same_orbit.period == "after"].nsmallest(
    post_event_scenes, "acquired"
)
sel = pd.concat([selected_before, selected_after]).sort_values("acquired")
sel = gpd.GeoDataFrame(sel, geometry="geometry", crs=products.crs)

print(f"Selected relative orbit {relative_orbit} ({pass_direction})")
if selected_after.empty:
    print("No post-event acquisition is available on this orbit yet. Rerun the search later.")
display(sel[["acquired", "period", "relativeOrbitNumber", "orbitDirection", "id"]])

sel.drop(columns="stac_item").to_file(
    output_root / "selected_products.geojson", driver="GeoJSON"
)
sel.drop(columns=["stac_item", "geometry"]).to_csv(
    output_root / "selected_products.csv", index=False
)

In [ ]:
m = explore_products(products=sel, aoi=shp)
serve_map(m)

In [ ]:
sel

## Download partial Sentinel-1 products

Set `RUN_DOWNLOAD` to `True` after checking the selected footprints. CDSE S3 credentials are expected in `/data/creds_s3.json`, matching the Berlin notebook. If there is no post-event scene yet, this downloads only the selected pre-event products; rerunning the search and download cells later will add post-event products without overwriting existing downloads.

In [ ]:
RUN_DOWNLOAD = True

if RUN_DOWNLOAD:
    if not credentials_file.exists():
        raise FileNotFoundError(f"CDSE S3 credentials not found: {credentials_file}")
    with credentials_file.open() as f:
        cred = json.load(f)

    download_partial_products(
        sel,
        shp,
        out_dir=data_dir,
        aws_key=cred["username"],
        aws_secret=cred["password"],
        pol="vv",
        force_overwrite=False,
    )
else:
    print("Dry run: set RUN_DOWNLOAD = True to download the selected partial products.")

In [ ]:
partial_products = sorted(data_dir.glob("*.partial.SAFE"))
print(f"Found {len(partial_products)} downloaded partial products in {data_dir}")
for product_dir in partial_products:
    print(product_dir.name)

## Process control and event InSAR pairs (VV)

Build one pre/pre control pair from the two selected pre-event acquisitions, then pair the closest pre-event acquisition with every selected post-event acquisition. Each pair produces geocoded VV primary/secondary amplitudes, coherence, and wrapped interferometric phase. Comparing the control and event pairs helps distinguish persistent terrain/geometry effects from event-driven change. Only downloaded products are processed.

In [ ]:
pre_rows = sel[sel.period == "before"].sort_values("acquired")
post_rows = sel[sel.period == "after"].sort_values("acquired")
if len(pre_rows) < 2:
    raise RuntimeError(f"Expected two selected pre-event scenes, found {len(pre_rows)}")

older_pre = pre_rows.iloc[-2]
nearest_pre = pre_rows.iloc[-1]
pair_records = [
    {
        "pair_type": "control_pre_pre",
        "prm_id": older_pre.id,
        "prm_acquired": older_pre.acquired,
        "sec_id": nearest_pre.id,
        "sec_acquired": nearest_pre.acquired,
    }
]
pair_records.extend(
    [
    {
        "pair_type": "event_pre_post",
        "prm_id": nearest_pre.id,
        "prm_acquired": nearest_pre.acquired,
        "sec_id": post_row.id,
        "sec_acquired": post_row.acquired,
    }
    for _, post_row in post_rows.iterrows()
    ]
)
pairs = pd.DataFrame(
    pair_records,
    columns=["pair_type", "prm_id", "prm_acquired", "sec_id", "sec_acquired"],
)

if post_rows.empty:
    print("No post-event scene is selected yet; only the control pair is available.")
pairs.to_csv(output_root / "insar_pairs.csv", index=False)
display(pairs)

Set `RUN_PROCESSING` to `True` to start processing. This can take substantial time. With `force_overwrite=False` in the download section, rerunning discovery and download later can add another post-event scene without replacing the existing partial products.

In [ ]:
from eo_tools.S1.process import process_insar

RUN_PROCESSING = True

if RUN_PROCESSING:
    if pairs.empty:
        print("Nothing to process: no selected post-event scene.")
    else:
        for pair in pairs.itertuples(index=False):
            prm_path = data_dir / f"{pair.prm_id}.partial.SAFE"
            sec_path = data_dir / f"{pair.sec_id}.partial.SAFE"
            missing = [path for path in (prm_path, sec_path) if not path.is_dir()]
            if missing:
                raise FileNotFoundError(
                    "Download the selected products first; missing: "
                    + ", ".join(str(path) for path in missing)
                )

            print(
                f"Processing {pair.pair_type}: "
                f"{pair.prm_acquired} -> {pair.sec_acquired}"
            )
            pair_dir = process_insar(
                prm_path=str(prm_path),
                sec_path=str(sec_path),
                output_dir=str(output_root),
                shp=shp,
                pol="vv",
                subswaths=["IW1", "IW2", "IW3"],
                write_coherence=True,
                write_interferogram=True,
                write_primary_amplitude=True,
                write_secondary_amplitude=True,
                apply_fast_esd=True,
                dem_name="cop-dem-glo-30",
                dem_upsampling=1.8,
                dem_force_download=False,
                dem_buffer_arc_sec=40,
                boxcar_coherence=[5, 5],
                filter_ifg=True,
                multilook=[2, 8],
                warp_kernel="bicubic",
                cal_type="beta",
                clip_to_shape=True,
                skip_preprocessing=False,
                orb_dir="/data/S1_orbits/",
            )
            print(f"Wrote outputs to {pair_dir}")
else:
    print("Dry run: set RUN_PROCESSING = True to process the pairs above.")

## Derive VV amplitude change

For each processed pair, calculate `20 × log10(after / before)` from the geocoded amplitudes. Positive values became brighter after the event; negative values became darker. A 5×5 rolling median reduces residual speckle. Invalid source pixels and the incomplete smoothing border are written as explicit nodata rather than zero.

In [ ]:
import numpy as np
import rioxarray as riox

def write_amplitude_change(pair_dir, window=5):
    pair_dir = Path(pair_dir)
    prm_file = pair_dir / "amp_prm_vv.tif"
    sec_file = pair_dir / "amp_sec_vv.tif"
    if not prm_file.exists() or not sec_file.exists():
        raise FileNotFoundError(f"Amplitude outputs are incomplete in {pair_dir}")

    prm = riox.open_rasterio(prm_file).squeeze("band", drop=True)
    sec = riox.open_rasterio(sec_file).squeeze("band", drop=True)
    valid = np.isfinite(prm) & np.isfinite(sec) & (prm > 0) & (sec > 0)
    change = 20 * np.log10(sec.where(valid) / prm.where(valid))
    if window and window > 1:
        change = change.rolling(
            x=window, y=window, center=True, min_periods=window * window
        ).median()
    change = change.where(valid).astype("float32").rename("amplitude_change_vv_db")
    nodata = -9999.0
    change = change.fillna(nodata)
    change.rio.write_nodata(nodata, inplace=True)

    out_file = pair_dir / "amplitude_change_vv_db.tif"
    change.rio.to_raster(
        out_file, driver="COG", compress="zstd", num_threads="all_cpus"
    )
    return out_file

pair_dirs = sorted(output_root.glob("S1_InSAR_*__*"))
if not pair_dirs:
    print("No processed pair directories found. Run the InSAR cell first.")
for pair_dir in pair_dirs:
    change_file = write_amplitude_change(pair_dir)
    print(f"Wrote {change_file}")

## Inspect the outputs

This map adds the before and after amplitudes, amplitude change, coherence, and wrapped phase as separate layers. Amplitude change uses a blue–white–red diverging palette centered on zero: blue is lower backscatter after the event and red is higher. Local COG display requires the TiTiler service used by the other demo notebooks.

In [ ]:
import folium
import json
import numpy as np
from matplotlib import colormaps
from matplotlib.colors import to_hex
from eo_tools_dev.util import palette_phi, show_cog

change_cmap = colormaps["RdBu_r"]
change_palette = json.dumps(
    {index: to_hex(change_cmap(index / 255)) for index in range(256)}
)

m = folium.Map(tiles=None)
folium.TileLayer(
    tiles=(
        "https://server.arcgisonline.com/ArcGIS/rest/services/"
        "World_Imagery/MapServer/tile/{z}/{y}/{x}"
    ),
    attr="Tiles &copy; Esri and contributors",
    name="Esri World Imagery",
    overlay=False,
    control=True,
).add_to(m)
layers = [
    ("amp_prm_vv.tif", "0,1", None),
    ("amp_sec_vv.tif", "0,1", None),
    ("amplitude_change_vv_db.tif", "-6,6", change_palette),
    ("coh_vv.tif", "0,1", None),
    ("phi_vv.tif", f"{-np.pi},{np.pi}", palette_phi()),
]

for pair_dir in sorted(output_root.glob("S1_InSAR_*__*")):
    for filename, rescale, colormap in layers:
        raster = pair_dir / filename
        if raster.exists():
            kwargs = {"rescale": rescale}
            if colormap is not None:
                kwargs["colormap"] = colormap
            show_cog(str(raster), m, **kwargs)

folium.LayerControl().add_to(m)
serve_map(m)